# Basics of Mobile Robotics - Project Report
### Prof. Francesco Mondada
###  EPFL - Fall 2025
###  Group 14
- <b> Jules VILLARET </b>(Global Navigation)
- <b> Tim KOBLER </b>(Filtering)
- <b> Amir LAHLOU </b>(Vision)
- <b> Ana SCHWABEDAL </b>(Local Navigation & Motion Control)

## Table of Contents
0. [Introduction](#introduction)
1. [Environment Design](#environment-design)
2. [Individual Modules](#individual-modules)
    - [Vision](#vision)
    - [Global Navigation](#global-navigation)
    - [Filtering](#filtering)
    - [Local Navigation](#local-navigation)
    - [Motion Control](#motion-control)
3. [Vision](#vision)
4. [Full System Integration](#full-system-integration)
5. [Conclusion](#conclusion)
6. [References](#references)


## Introduction
Imagine a small robot casually exploring the surface of the Moon. A dust storm is approaching, and the robot must autonomously navigate through craters and rock to return safely to its lander. To succeed, it must perceive its surroundings, estimate its position despite uncertainty, plan a safe route, and react instantly to unexpected obstacles.

In this project, we recreate this scenario using the Thymio robot. Our goal is to design a complete navigation system that allows the robot to move reliably through a predefined map, reach a target placed anywhere in the environment and avoid both known and unexpected obstacles. The Thymio relies on an overhead camera for global pose estimation, an Extended Kalman Filter to correct odometry drift, and proximity sensors for reactive obstacle avoidance.

This report presents the main design choices and algorithms used to achieve these tasks. It follows the structure recommended in the MICRO-452 Basics of Mobile Robotics course taught by Prof. Francesco Mondada, and details the different modules of our system: computer vision, global navigation, filtering, local navigation and motion control.

## Demonstration Video
*Below is a visualization of the robot's movement accross the environment.*


[![Video preview](Images%20and%20Videos/thumbnail.png)](Images%20and%20Videos/demonstration.mp4)

---
## 1. Vision 
### Using the OpenCV library for detection & localisation
---
## 1.1 Obstacle Detection
### 1.1.1 Initialization

In order to detect obstacles, the following steps are implemented :
1. **Conversion to grayscale :** In order to simplify computations, unnecessary color information is removed.
2. **Gaussian Blur :** Smoothens the image in order to reduce the noise inherent to teh camera sensor.
3. **Binarisation (Inverted Threshold) :** 
    * Light pixels (**intensity 100 to 255**) become black **(intensity = 0)**
    * Dark pixels (**intensity 0 to 100**) become white **(intensity = 1)**

    The rationale behind the step 3 is the fact that the contour detection algorith needs the resulting image to be binary, containing only 1's and 0's, where 1 corresponds to an obstacle. 

> **Limitations and Constraints :**
> * **Environment :** The Thymio as well as the goal and any irrelevant objects must be out of the frame when performing the initial capture to avoid skewing the analysis. 
> * **Shadows :** Strong shadows may mistakingly be interpreted as dark areas, therefore as an obstacle, which creates a false positive.


In [3]:
#mettre les 3 image: image original, image niveaux de gris+flou , image apres seuillage

### Edge Detection

Extracting the geometry of the obstacles is done by using the `cv2.findContours` function.

* **How it works :** This function analyzes the binary image and returs a list of coordinates $(x, y)$ which draw the shapes. 
* **The logic behind it :** The algorithm detects borders by identifying transition zones. In other words, it identifies the pixels where the shades transition from white (where the value is 0) to black (where the value is 1).The interface of neighboring black and white pixels thus draws the edges of the obstacles.

### 1.1.2 Corner Detection 

In order to identify the significant corners of a contour, the `cv2.approxPolyDP` function is used. This function implements the **Ramer–Douglas–Peucker algorithm**.

**Principle of the algorithm:**

The objective is to simplify the shape of the contour by reducing the number of points while preserving its overall structure (the corners).

**Step 1, Initialization:** The algorithm draws a segment connecting the first and last points of the detected contour.
**Step 2, Searching for the critical point:** The next step is to identify the point that is the furthest away from this segment.
**Step 3, Thresholding ($\epsilon$):** 
- If the distance to the segment is greater than a pre-defined threshold, noted $\epsilon$, the point is considered “important” (in other words, it is a potential corner). 
- The contour is then split into two new segments that pass through this point, and the procedure is applied recursively to each of them.
- If the distance is below $\epsilon$, the point (and the intermediate segments) are ignored.

**Final result:**
The function returns a reduced list of points. These points correspond to the vertices of the simplified polygon, ie. they represent the corners of the obstacles.

*Figure 2: Ramer–Douglas–Peucker algorithm*

![My image](Images%20and%20Videos/250px-Douglas_Peucker.png)

The figure shows progressive simplification of a contour by recursively removing points whose deviation is below the threshold $\epsilon$. Thus illustrating how the algorithm keeps only the points that significantly change the contour’s shape, reducing noise while preserving the main geometry.

In [4]:
# IMAGE 4 COINS DES OBSTACLES

## 1.2 Localisation and Homography
This section establishes the connection between the camera (pixels) and the real world (meters).

### 1.2.1 ArUco Markers

**ArUco marker detection algorithm**

The `ArUco` library (integrated into OpenCV) allows us to precisely locate the corners of ArUco markers in the image and to decode their unique identifier (ID). This makes it possible to distinguish multiple markers within the same environment. The implemented algorithms are robust and follow the procedure below:

1. **Candidate Extraction:**
The system first applies image processing similar to contour detection (see Section 1.1) to identify closed quadrilateral shapes. Those are the potential candidates.

2. **Rectification (Canonical Form):**
When a 4-corner quadrilateral is detected, a homography transformation is applied to “straighten” the marker and obtain a front-facing square view.

3. **Binarization and Reading:**
Then, the rectified region is divided into a grid of bits to extract the marker’s binary code.

4. **Identification and Rotation:**
Finally, this binary code is compared against the library of known ArUco markers which is a dictionary.
- The algorithm tests the 4 possible rotations of the code to compensate for marker orientation.
- It computes the **Hamming distance**. This is the sum of the differences between the read bits and those that are in the dictionary.

5. **Validation:**
If this distance is below a tolerance threshold, the marker is validated. The system then returns the marker’s ID as well as the $(u, v)$ pixel coordinates of its four corners in the image.

**Role of the ArUco Markers**
In the proposed configuration, the ArUco markers are indispensible for obtaining the geometry of the scene:

1. **Environment Delimitation:** Fixed markers define the corners or edges of the arena, allowing the area that can be navigated to be calculated.
2. **Tracking:** The markers are used to obtain the position $(x, y)$ and orientation of the Thymio robot, as well as the coordinates of the target to reach.

### 1.2.2. Homography

To establish the correspondence between the image and the real world, we compute a **homography matrix**.

1. **Matrix Computation**

We use the **inner corners** of the 4 ArUco markers placed at the boundaries of the arena as reference points. These points allow us to define the ground plane.

- **World Frame:**  
  We arbitrarily define the origin $(0, 0)$ at the **outer corner** of the bottom-left ArUco marker.

2. **Purpose of the Transformation**

This matrix **H** converts coordinates from the **Camera** frame (pixels) to the **Environment** frame (millimeters).  
It serves three essential functions:

1. **Image Rectification (Warping):**  
   Extracts only the interior region of the arena (excluding boundary ArUcos), transforms it to a top-down view, and provides this cleaned image to the obstacle-detection algorithm (Section 1).

2. **Localization:**  
   Converts the positions of the robot and the target detected in the raw image into their real coordinates in the environment.

3. **Display (Augmented Reality):**  
   Allows drawing environment-frame points directly onto the camera feed (e.g., overlaying the path to follow).

## Mathematical Definition

A homography is a projective transformation represented by a **3 × 3 matrix**.  
It maps a source point \(P_{\text{src}}(u, v)\) in the image to a destination point \(P_{\text{dst}}(x, y)\) in the rectified plane according to:

```math
s
\begin{bmatrix}
x \\
y \\
1
\end{bmatrix}
=
H
\begin{bmatrix}
u \\
v \\
1
\end{bmatrix}
=
\begin{bmatrix}
h_{11} & h_{12} & h_{13} \\
h_{21} & h_{22} & h_{23} \\
h_{31} & h_{32} & h_{33}
\end{bmatrix}
\begin{bmatrix}
u \\
v \\
1
\end{bmatrix}


---
## 2. Global Navigation
### Planning collision-free paths using a visibility-graph approach
---
### 2.1 Objective

The role of the global navigation module is to compute a collision-free path from the robot’s current position to a target that can be placed anywhere on the map. In our setup, the environment consists two polygonal obstacles. The Thymio should avoid these obstacles without using its proximity sensors, meaning the global planner must ensure that the computed path is valid only based on the map.

To achieve this, we implemented a visibility-graph based global planner combined with Dijkstra’s shortest-path algorithm. The following libraries are required for numerical operations, graph computation, geometric calculations, and visualization:

In [ ]:
import numpy as np
import heapq
from math import sqrt
import constants as cst
import matplotlib.pyplot as plt

class GlobalNav:

    def __init__(self, arena_width=1189, arena_height=841):
        self.width = arena_width
        self.height = arena_height
        self.nodes = []
        self.edges = {}

### Why a Visibility Graph?

A visibility graph is a classical solution for motion planning in polygonal environments. It has several advantages relevant to our project:
- It guarantees optimal paths (shortest Euclidean distance).
- It works well with polygonal obstacles.
- It produces smooth global paths.
- It is simple enough to implement and debug, aligning with the course recommendation to keep the project as simple as possible.

The idea is to connect all obstacle vertices, the start, and the goal with straight segments whenever the segment is not obstructed by any polygon.

### Obstacle Inflation

One practical concern is that the Thymio has a non-zero radius (~55 mm). A path that technically “touches” an obstacle vertex could still lead to a collision. To avoid this, we inflate each obstacle by a constant inflation radius `INFLATION_R` of 140 mm :
- Compute each polygon’s centroid.
- Move every vertex outward along the direction of the centroid-to-vertex vector.

This ensures that:
- The visibility graph is computed around slightly “larger” obstacles.
- Any path found is safe for the physical robot.

This step is essential for creating a safe clearance around obstacles without requiring complex configuration-space geometry. You can find below the fucntion `inflate_polygon()`, who takes the list of the polygon vertices as input and outpouts the inflated vertices. 

In [ ]:
def inflate_polygon(self, poly): #inflate a polygon by moving each corner away from its centroid
        cx = np.mean([p[0] for p in poly]) #finfing centroid
        cy = np.mean([p[1] for p in poly])

        inflated = []
        for (x, y) in poly:
            vx = x - cx #dist from centroid to corner
            vy = y - cy
            L = sqrt(vx * vx + vy * vy) + 1e-6 #avoid div by 0, when center is on a point(like a 3 point line)
            scale = (L + cst.INFLATION_R) / L #scale factor to push vertex outward by inflation radius
            inflated.append((cx + vx * scale, cy + vy * scale)) #apply scaling to get inflated vertex
        return inflated

### Checking Segment Visibility
Two nodes A and B are connected by an edge only if segment AB is fully visible, meaning :
1. AB does not intersect any obstacle edge.
2. The midpoint of AB is not inside any inflated polygon (important when A or B lies directly on the obstacle boundary).

To determine whether two points were mutually visible, we relied on a few standard geometric tests such as checking segment intersections and detecting whether a point lies inside a polygon. These are classical tools used in computational geometry and widely applied in robot navigation.

- `ccw()` : Determines if three points are counter-clockwise, a prerequisite for segment intersection.
- `segments_intersect()` : Two segments AB and CD intersect if C and D lie on different sides of AB and A and B lie on different sides of CD.
- `segment_hits_polygon()` : Checks if the segment AB intersects any edges of the selected polygon.
- `point_in_polygon()` (ray casting) : Used to detect cases where a segment passes through a polygon without intersecting its edges.

These geometric tests ensure the correctness of the visibility graph in environments with arbitrary convex or concave shapes.

In [ ]:

def ccw(self, A, B, C): #are the points in counter-clockwise order? return True if > 0, False for cw < 0 and colinear = 0
    return (C[1] - A[1]) * (B[0] - A[0]) - (B[1] - A[1]) * (C[0] - A[0]) > 0 #https://www.geeksforgeeks.org/dsa/orientation-3-ordered-points/

def segments_intersect(self, p1, p2, p3, p4): #return True if they cross
    return (self.ccw(p1, p3, p4) != self.ccw(p2, p3, p4)) and (self.ccw(p1, p2, p3) != self.ccw(p1, p2, p4)) #https://www.geeksforgeeks.org/dsa/check-if-two-given-line-segments-intersect/
    #two segments AB and CD intersect if and only if: C and D are on different sides of AB and A and B are on different sides of CD

def segment_hits_polygon(self, A, B, poly): #true if segment AB intersects any edge of polygon poly
    n = len(poly)
    for i in range(n):
        C = poly[i]
        D = poly[(i + 1) % n] #pair of edges, loops around

        #if AB shares an endpoint with the edge CD -> allowed
        if A == C or A == D or B == C or B == D:
            continue #skip next if

        #check edge intersection
        if self.segments_intersect(A, B, C, D):
            return True  
    return False

def point_in_polygon(self, point, poly):
    x, y = point
    inside = False
    n = len(poly)

    for i in range(n):
        x1, y1 = poly[i]
        x2, y2 = poly[(i + 1) % n]

        #cross product == 0 -> collinear
        cross = (x - x1) * (y2 - y1) - (y - y1) * (x2 - x1)
        if abs(cross) < cst.EPSILON: #accept tiny tolerance because never 0
            # dot product <= 0 -> point is between endpoints
            dot = (x - x1) * (x - x2) + (y - y1) * (y - y2)
            if dot <= cst.EPSILON:
                return False  #on edge = considered outside

        #Ray casting : check if the horizontal ray crosses edge (x1,y1)-(x2,y2)
        if (y1 > y) != (y2 > y): #y is between y1 and y2; compute intersection x
            x_intersect = (x2 - x1) * (y - y1) / (y2 - y1) + x1
            if x < x_intersect:
                inside = not inside
    return inside #True = inside, False = outside

def segment_visible(self, A, B, polygon_list):
    mid = ((A[0] + B[0]) / 2.0, (A[1] + B[1]) / 2.0)

    for poly in polygon_list:
        if self.segment_hits_polygon(A, B, poly): #AB must not intersect any polygon edge
            return False
        if self.point_in_polygon(mid, poly): #midpoint inside polygon (for segments between the polygon's points)
            return False
    return True

### Building the Visibility Graph

The construction follows these steps:
1. Add start and goal points.
2. Add all inflated obstacle vertices.
3. For every pair of nodes (A, B):
    - Check whether segment AB is visible.
    - If yes, add an undirected edge with weight equal to the Euclidean distance in `edges`.

This produces:
- A list of `nodes` (coordinates in mm),
- A dictionary of `edges`, where each key is a node index and its value is a list of neighbors with associated distances.

This representation is suitable for all classical graph search algorithms.

In [ ]:
def build_visibility_graph(self, start, goal, obstacles): #build the visibility graph on inflated obstacles
    """
    Inputs:
        start, goal: (x, y) points in mm
        obstacles: list of polygons [[(x,y),...], [...]]
    Returns:
        nodes: list of all points (start, goal, obstacle vertices)
        edges: dict {idx: [(neighbor_idx, distance), ...]}
    """
    inflated = [self.inflate_polygon(poly) for poly in obstacles] #inflate obstacles

    nodes = [start, goal] #idx 0 = start, 1 = goal
    for poly in inflated: #collect all nodes
        nodes.extend(poly)

    #build edges between all visible pairs
    N = len(nodes)
    edges = {i: [] for i in range(N)}

    for i in range(N):
        for j in range(i + 1, N):
            A = nodes[i]
            B = nodes[j]

            if self.segment_visible(A, B, inflated):
                d = self.dist(A, B)
                edges[i].append((j, d))
                edges[j].append((i, d)) #adds to list with [(point, distance),...]

    return nodes, edges

### Shortest Path Computation

Once the visibility graph is constructed, we compute the shortest path using Dijkstra’s algorithm, implemented using a priority queue (<b>heapq</b>):
- Initialize with cost(start) = 0
- Pop the node with minimal accumulated distance
- Update neighbors if a shorter path is found
- Continue until reaching the goal

We then reconstruct the path using the `parent` dictionary.

Why Dijkstra?
- All edge weights are positive → Dijkstra is optimal.
- Simpler than A*.
- Meets the course requirement of maintaining simplicity.

The final output is a list of nodes going from start to goal.

In [ ]:
def dijkstra(self, edges, start_idx, goal_idx): #help from Exercise 5 part 1
    pq = [(0.0, start_idx)] #(cost_so_far, node_index), priority queue implemented with heapq
    cost = {start_idx: 0.0} #dictionary: node -> best known cost
    parent = {start_idx: None} #keeps track of how we reached each node

    while pq:
        c, u = heapq.heappop(pq) #pops the element with smallest cost, u=node and c=cost
        if u == goal_idx:
            break

        for v, w in edges[u]: #explore neighbors of u
            newc = c + w #total cost to reach v via u
            if v not in cost or newc < cost[v]: #update if first time visiting v or found better path
                cost[v] = newc
                parent[v] = u
                heapq.heappush(pq, (newc, v))

    if goal_idx not in parent: #if goal has no parent, no path exists
        return None  # no path

    #reconstruct path, go backwards using the parent dictionary
    path = []
    cur = goal_idx
    while cur is not None:
        path.append(cur)
        cur = parent[cur]
    return path[::-1] #returns list of node indices for the shortest path.

### Path Output
The function `find_optimal_path()` returns:
- A list of waypoints from the robot’s current pose to the goal.
- These waypoints are then fed into the motion controller, which tracks the path segment by segment.

If no valid path exists, the function safely reports that no solution was found.

In [ ]:
def find_optimal_path(self, thymio, goal, obstacles):
    """
    Inputs:
        thymio: (x, y, theta) in mm, rad
        goal: (x, y) in mm
        obstacles: list of polygons from vision
    Returns:
        path_world: list of (x, y) waypoints in mm.
    """
    start_pt = [thymio[0], thymio[1]]
    goal_pt = [goal[0], goal[1]]

    self.nodes, self.edges = self.build_visibility_graph(start_pt, goal_pt, obstacles)

    path_idx = self.dijkstra(self.edges, start_idx=0, goal_idx=1) #goal idx = 1 bc of build_visibility
    if path_idx is None:
        print("GlobalNav (visibility): No path found!")
        return []

    path_world = [self.nodes[i] for i in path_idx] #convert node indices to (x,y) coordinates
    return path_world


### Goal Reached?

To determine whether the Thymio has reached the goal, we use a simple distance-based criterion. Since the robot is not a point, we define a tolerance region around the goal `GOAL_TOL = 20`. If the robot’s current position lies within this radius, we consider the goal reached and stop global navigation. This is checked using the Euclidean distance.

In [ ]:
def is_goal_reached(self, current_pos, goal_pos): #tol = 70mm bc R of the Thymio is ~55, so a bit more 
    return self.dist(current_pos, goal_pos) < cst.GOAL_TOL


### Map Visualization
To help with debugging and for presentation purposes, we added a plotting function that displays:
- Original obstacles
- Inflated obstacles (implicitly through edges)
- All visibility edges
- All nodes
- The optimal path highlighted in red
- Start and goal points

The axes are scaled to match the arena dimensions (1189×841 mm) and oriented so that the origin matches our coordinate convention.

This visualization was extremely helpful to verify:
- Whether obstacle inflation was correct
- Whether occluded edges were properly removed
- Whether Dijkstra returned the expected path

It also provides a clear illustration for the final demo.

In [ ]:
# to add ![visibility_graph.png](attachment:visibility_graph.png)

---
## Filtering
### Estimating robot pose with an Extended Kalman Filter
---

To ensure robust position estimation of the Thymio robot in the environment, we implemented an **Extended Kalman Filter (EKF)** that fuses data from two sources:
- Visual measurements from the camera (position x, y and orientation θ)
- Motor commands and odometry (left and right wheel speeds)

The filter compensates for uncertainties in the process and the measurements and ensures continuous position estimation even when the camera temporarily loses sight of the robot.

### State Model

The kinematic model of the robot can be defined in several different ways.
1. We can consider the robot's position $(x, y, \theta)$ as system states and velocity as inputs. With $A$ being a 3x3 identity matrix, $B$ the control matrix, and $u$ the wheel speeds.
$$\mathbf{x}_{k+1} = \mathbf{A} \mathbf{x}_{k} + \mathbf{B} \mathbf{u}_{k}$$

2. We can also consider velocity as a system state, without external inputs, thus allowing for robot velocity estimation in addition to position. This improves the overall precision of the Kalman filter. $A$ is therefore the 5x5 state transition matrix.
$$\mathbf{x}_{k+1} = \mathbf{A} \mathbf{x}_{k}$$


After discussing with professor Ferrari Trecate Giancarlo giving the multivariable control course (ME-422), we chose to prioritize the second approach by defining the states as follows:

$$\mathbf{x} = \begin{bmatrix} x \\ y \\ \theta \\ v_L \\ v_R \end{bmatrix}$$

where:
- $x, y$ : robot position in mm
- $\theta$ : robot orientation in radians (positive counterclockwise)
- $v_L, v_R$ : left and right wheel speeds in mm/s

### Kinematic Model

The differential robot's kinematic model can be described by the following equations:

$$\mathbf{x}_{k+1} = g(\mathbf{x}_k) = \begin{bmatrix}
x_k + v_{L_k}\Delta t\frac{\cos(\theta)}{2} + v_{R_k}\Delta t\frac{\cos(\theta)}{2} \\
y_k + v_{L_k}\Delta t\frac{\sin(\theta)}{2} + v_{R_k}\Delta t\frac{\sin(\theta)}{2} \\
\theta_k - \Delta t\frac{v_{L_k}}{L} + \Delta t\frac{v_{R_k}}{L}\\
v_{L_k} \\
v_{R_k}
\end{bmatrix} + \mathbf{W}_{k}$$

$$\mathbf{y}_{k} = \mathbf{C} \mathbf{x}_{k} + \mathbf{V}_{k}$$

where:
- $\Delta t$ : time step between two measurements
- $L = 122.7$ mm : robot wheelbase
- $C$ : observation matrix (identity matrix 5x5)
- $W$ : process noise (Gaussian white noise with covariance $Q$)
- $V$ : measurement noise (Gaussian white noise with covariance $R$)

This translates to the following non-linear state transition matrix:
$$\mathbf{A} = \begin{bmatrix}
1 & 0 & 0 & \frac{\Delta t}{2}\cos(\theta) & \frac{\Delta t}{2}\cos(\theta) \\
0 & 1 & 0 & \frac{\Delta t}{2}\sin(\theta) & \frac{\Delta t}{2}\sin(\theta) \\
0 & 0 & 1 & -\frac{\Delta t}{L} & \frac{\Delta t}{L} \\
0 & 0 & 0 & 1 & 0 \\
0 & 0 & 0 & 0 & 1
\end{bmatrix}$$

It can be noted that the model is non-linear due to the trigonometric terms involving the angle $\theta$.
It is therefore necessary to use an Extended Kalman Filter (EKF) to handle this non-linearity. We proceed by linearizing the model at each step around the current state to then work on an approximated linear model. The Jacobian is given by the state transition matrix $J$.

$$\mathbf{J} = \frac{\delta g(\mathbf{x}_k)}{\delta \mathbf{x}_k} = \begin{bmatrix}
1 & 0 & -\sin(\theta)\Delta t\frac{v_L-v_R}{2} & \frac{\Delta t}{2}\cos(\theta) & \frac{\Delta t}{2}\cos(\theta) \\
0 & 1 & \cos(\theta)\Delta t\frac{v_L-v_R}{2} & \frac{\Delta t}{2}\sin(\theta) & \frac{\Delta t}{2}\sin(\theta) \\
0 & 0 & 1 & -\frac{\Delta t}{L} & \frac{\Delta t}{L} \\
0 & 0 & 0 & 1 & 0 \\
0 & 0 & 0 & 0 & 1
\end{bmatrix}$$

The Jacobian is used in the calculation step of the predicted state covariance.

### Covariance Matrices
#### Matrix Q - Process Noise
The $Q$ matrix represents the uncertainty in the motion model:

$$\mathbf{Q} = \begin{bmatrix}
Qx & 0 & 0 & 0 & 0 \\
0 & Qy & 0 & 0 & 0 \\
0 & 0 & Q\theta & 0 & 0 \\
0 & 0 & 0 & Qv_L & 0 \\
0 & 0 & 0 & 0 & Qv_R
\end{bmatrix}$$

The values reflect:
- $Qx$, $Qy$: Position uncertainty: 3.2 mm² in x and y
- $Q\theta$: Orientation uncertainty: 0.214 rad²
- $Qv_L$, $Qv_R$: Speed uncertainty: 0.57 and 0.53 (mm/s)²

The state variances were determined empirically based on behaviors observed during robot tests.
- To determine the variances $Qx$ and $Qy$, we drove the robot in a straight line at constant speed command for 50cm and calculated the standard deviation of the traveled distance measurements.
- To determine the variance $Q\theta$, we rotated the robot on the spot at constant speed command for a 360° angle and calculated the standard deviation of the resulting angle measurements.
- To determine the variances $Qv_L$ and $Qv_R$, we calculated the standard deviation of the robot's speed by dividing the distance covered by the measured time. This was done while the robot was travelling at a constant speed over a fixed distance.

#### Matrix R - Measurement Noise
The **R** matrix adapts according to camera availability:

**With camera**:
$$\mathbf{R}_{camera} = \begin{bmatrix}
Rx & 0 & 0 & 0 & 0 \\
0 & Ry & 0 & 0 & 0 \\
0 & 0 & R\theta & 0 & 0 \\
0 & 0 & 0 & Rvl & 0 \\
0 & 0 & 0 & 0 & Rvr
\end{bmatrix}$$

The values are set as follows:
- $Rx$, $Ry$: Position uncertainty from camera measurements: e.g., 0.349 mm² in x and y
- $R\theta$: Orientation uncertainty from camera measurements: e.g., 0.027 rad²
- $Rvl$, $Rvr$: Wheel speed measurement uncertainty: e.g., 3.46 and 5.05 (mm/s)² (respectively for left and right wheels)

These values are also determined based on sensor characteristics and empirical tuning.
- To determine the variances $Rx$, $Ry$, and $R\theta$, we calculated the standard deviation of the positions measured by the camera with a stationary robot. We placed the robot in a corner of the map to ensure maximum measurement distortion and take the maximal value of standard deviation.
- To determine the variances $Rvl$ and $Rvr$, we calculated the standard deviation of the speeds measured by the robot (on the left and right wheels) during a straight-line run at constant speed command.


**Without camera**:

When the camera is unavailable, position uncertainties are set to infinity, forcing the filter to ignore these measurements and rely only on odometry.
$$\mathbf{R}_{no\ camera} = \begin{bmatrix}
\infty & 0 & 0 & 0 & 0 \\
0 & \infty & 0 & 0 & 0 \\
0 & 0 & \infty & 0 & 0 \\
0 & 0 & 0 & Rvl & 0 \\
0 & 0 & 0 & 0 & Rvr
\end{bmatrix}$$

### Kalman Filter Algorithm

The filter follows the classical Kalman filter steps at each iteration:

#### 1. Prediction
A Priori state and variance estimate
$$\hat{\mathbf{x}}_{k} = \mathbf{A} \mathbf{x}_{k-1}$$
$$\hat{\mathbf{P}}_{k} = \mathbf{J} \mathbf{P}_{k-1} \mathbf{J}^T + \mathbf{Q}$$

#### 2. Innovation
$$\mathbf{i}_k = \mathbf{y}_k - \mathbf{C} \hat{\mathbf{x}}_{k}$$
$$\mathbf{S}_k = \mathbf{C} \hat{\mathbf{P}}_{k} \mathbf{C}^T + \mathbf{R}$$
where $\mathbf{C} = \mathbf{I}_5$ (identity observation matrix since all state components are measured).

#### 3. Kalman Gain
$$\mathbf{K}_k = \hat{\mathbf{P}}_{k} \mathbf{C}^T \mathbf{S}_k^{-1}$$
#### 4. Estimation Update
A Posteriori state and variance estimate
$$\mathbf{x}_{k} = \hat{\mathbf{x}}_{k} + \mathbf{K}_k \mathbf{i}_k$$
$$\mathbf{P}_{k} = (\mathbf{I} - \mathbf{K}_k \mathbf{C}) \hat{\mathbf{P}}_{k}$$

### Implementation

The `Filter` class encapsulates the Kalman filter implementation. It is initialized with the initial position measured by the camera:

```python
filter = Filter(thymio)  # thymio contains [x, y, theta] and are init with a camera measurement
```

At each iteration, the `update()` method is called with:
- `measured_pos` : position measured by the camera (or `None` if unavailable)
- `Ts` : time elapsed since the last update
- `ls, rs` : measured left and right wheel speeds

The filter returns the estimated position `[x, y, theta]` which is used for robot navigation.

## References

For the integration of this part, we used the listed references to guide us more into our work. However, note that the full code was all written by ourselves, taking into account the information provided by the following referenced websites:
- https://www.geeksforgeeks.org/dsa/orientation-3-ordered-points/
- https://www.geeksforgeeks.org/dsa/check-if-two-given-line-segments-intersect/
- https://chatgpt.com
- Exercise session in class